<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/movie_recommendation_system_Collaborative_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaborative Filtering

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
DATASET_PATH = "/content/drive/MyDrive/ml-32m/ml-32m"

In [12]:
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
user_item_sparse = load_npz(
    "/content/drive/MyDrive/Artifact/user_item_sparse.npz"
)

user_encoder = joblib.load(
    "/content/drive/MyDrive/Artifact/user_encoder.pkl"
)

movie_encoder = joblib.load(
    "/content/drive/MyDrive/Artifact/movie_encoder.pkl"
)

movie_data = joblib.load(
    "/content/drive/MyDrive/Artifact/movie_data.pkl"
)

In [ ]:
print(user_item_sparse.shape)

(200948, 84432)


In [ ]:
svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

user_features = svd.fit_transform(user_item_sparse)

movie_features = svd.components_.T

In [ ]:
print(
    f"Explained Variance: "
    f"{svd.explained_variance_ratio_.sum():.4f}"
)

Explained Variance: 0.4091


In [ ]:
joblib.dump(
    svd,
    "/content/drive/MyDrive/Artifact/svd_model.pkl"
)

joblib.dump(
    user_features,
    "/content/drive/MyDrive/Artifact/user_features.pkl"
)

joblib.dump(
    movie_features,
    "/content/drive/MyDrive/Artifact/movie_features.pkl"
)

['/content/drive/MyDrive/Artifact/movie_features.pkl']

In [13]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/ml-32m/ml-32m/ratings.csv"
)

In [14]:
def recommend_collaborative(user_id, top_n=10):

    if user_id not in user_encoder.classes_:
        print("User not found.")
        return None

    user_index = user_encoder.transform([user_id])[0]

    user_vector = user_features[user_index].reshape(1, -1)

    scores = user_vector @ movie_features.T

    scores = scores.flatten()

    watched_movies = ratings.loc[
        ratings["userId"] == user_id,
        "movieId"
    ]

    watched_indices = movie_encoder.transform(
        watched_movies[
            watched_movies.isin(movie_encoder.classes_)
        ]
    )

    scores[watched_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:top_n]

    recommended_movie_ids = movie_encoder.inverse_transform(
        top_indices
    )

    recommendations = movie_data[
        movie_data["movieId"].isin(recommended_movie_ids)
    ].copy()

    recommendations["predicted_score"] = [
        scores[idx] for idx in top_indices
    ]

    return recommendations.sort_values(
        "predicted_score",
        ascending=False
    )

In [15]:
recommend_collaborative(
    user_id=1,
    top_n=10
)

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,predicted_score
734,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,1964.0,Dr. Strangelove or: How I Learned to Stop Worr...,"[Comedy, War]",57012,935.0,black and white stanley kubrick classic dark c...,Dr. Strangelove or: How I Learned to Stop Worr...,2.863783
840,858,"Godfather, The (1972)",Crime|Drama,1972.0,"Godfather, The","[Crime, Drama]",68646,238.0,al pacino atmospheric great acting masterpiece...,"Godfather, The Crime Drama Al Pacino atmospher...",2.771993
903,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,1968.0,2001: A Space Odyssey,"[Adventure, Drama, Sci-Fi]",62622,62.0,atmospheric space stanley kubrick ambiguous at...,2001: A Space Odyssey Adventure Drama Sci-Fi a...,2.545833
1176,1207,To Kill a Mockingbird (1962),Drama,1962.0,To Kill a Mockingbird,[Drama],56592,595.0,tumey s dvds book was better classic harper le...,To Kill a Mockingbird Drama Tumey's DVDs book ...,2.479119
1191,1222,Full Metal Jacket (1987),Drama|War,1987.0,Full Metal Jacket,"[Drama, War]",93058,600.0,vietnam war tumey s dvds imdb top 250 stanley ...,Full Metal Jacket Drama War Vietnam War Tumey'...,2.291048
1198,1230,Annie Hall (1977),Comedy|Romance,1977.0,Annie Hall,"[Comedy, Romance]",75686,703.0,tumey s dvds woody allen funny overrated sarca...,Annie Hall Comedy Romance Tumey's DVDs Woody A...,1.958763
1217,1250,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,1957.0,"Bridge on the River Kwai, The","[Adventure, Drama, War]",50212,826.0,tumey s dvds classic oscar best picture world ...,"Bridge on the River Kwai, The Adventure Drama ...",1.841476
1273,1307,When Harry Met Sally... (1989),Comedy|Romance,1989.0,When Harry Met Sally...,"[Comedy, Romance]",98635,639.0,romantic comedy best friends fall in love bill...,When Harry Met Sally... Comedy Romance romanti...,1.717384
1559,1617,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller,1997.0,L.A. Confidential,"[Crime, Film-Noir, Mystery, Thriller]",119488,2118.0,kevin spacey conspiracy crime femme fatale hol...,L.A. Confidential Crime Film-Noir Mystery Thri...,1.715159
2766,2858,American Beauty (1999),Drama|Romance,1999.0,American Beauty,"[Drama, Romance]",169547,14.0,bittersweet great acting kevin spacey loneline...,American Beauty Drama Romance bittersweet grea...,1.674148


In [16]:
recommend_collaborative(5)
recommend_collaborative(20)
recommend_collaborative(100)

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,predicted_score
362,367,"Mask, The (1994)",Action|Comedy|Crime|Fantasy,1994.0,"Mask, The","[Action, Comedy, Crime, Fantasy]",110475,854.0,jim carrey comedy jim carrey cartoonish jim ca...,"Mask, The Action Comedy Crime Fantasy Jim Carr...",2.848728
534,539,Sleepless in Seattle (1993),Comedy|Drama|Romance,1993.0,Sleepless in Seattle,"[Comedy, Drama, Romance]",108160,858.0,overrated comedy meg ryan romance romantic com...,Sleepless in Seattle Comedy Drama Romance over...,2.815555
536,541,Blade Runner (1982),Action|Sci-Fi|Thriller,1982.0,Blade Runner,"[Action, Sci-Fi, Thriller]",83658,78.0,cyberpunk dystopia existentialism neo noir phi...,Blade Runner Action Sci-Fi Thriller cyberpunk ...,2.260608
589,597,Pretty Woman (1990),Comedy|Romance,1990.0,Pretty Woman,"[Comedy, Romance]",100405,114.0,capitalism happy ending rags to riches romance...,Pretty Woman Comedy Romance capitalism happy e...,2.215445
898,919,"Wizard of Oz, The (1939)",Adventure|Children|Fantasy|Musical,1939.0,"Wizard of Oz, The","[Adventure, Children, Fantasy, Musical]",32138,630.0,based on a book classic fantasy heartwarming m...,"Wizard of Oz, The Adventure Children Fantasy M...",2.207534
1062,1089,Reservoir Dogs (1992),Crime|Mystery|Thriller,1992.0,Reservoir Dogs,"[Crime, Mystery, Thriller]",105236,500.0,dark comedy dialogue one location movie one ro...,Reservoir Dogs Crime Mystery Thriller dark com...,1.988696
1640,1704,Good Will Hunting (1997),Drama|Romance,1997.0,Good Will Hunting,"[Drama, Romance]",119217,489.0,thought provoking emotional predictable simple...,Good Will Hunting Drama Romance thought-provok...,1.872128
1923,2012,Back to the Future Part III (1990),Adventure|Comedy|Sci-Fi|Western,1990.0,Back to the Future Part III,"[Adventure, Comedy, Sci-Fi, Western]",99088,196.0,alan silvestri time travel christopher lloyd e...,Back to the Future Part III Adventure Comedy S...,1.863673
3846,3949,Requiem for a Dream (2000),Drama,2000.0,Requiem for a Dream,[Drama],180093,641.0,atmospheric dark depressing disturbing emotion...,Requiem for a Dream Drama atmospheric dark dep...,1.849277
20255,104841,Gravity (2013),Action|Sci-Fi|IMAX,2013.0,Gravity,"[Action, Sci-Fi, IMAX]",1454468,49047.0,cheesy disappointing oscar best sound editing ...,Gravity Action Sci-Fi IMAX cheesy disappointin...,1.768136
